b) Practical / Coding

Write a PySpark Structured Streaming program to:

- Read data from a streaming source (e.g., socket or file directory)

- Perform a simple transformation (such as word count)

- Output the results to the console

- Explain each step in comments Provide correct answers

In [1]:
pip install pyspark findspark --no-cache-dir

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split
import threading
import socket
import time

# Initialize Spark Session
# master("local[2]") is REQUIRED for streaming (1 thread to receive, 1 to process)
spark = SparkSession.builder \
    .appName("Assignment3") \
    .master("local[2]") \
    .getOrCreate()

# Set log level to ERROR to keep the output clean
spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [2]:
def start_mock_server():
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_socket.bind(('localhost', 9999))
    server_socket.listen(1)
    print("[SERVER] Waiting for Spark to connect...")

    conn, addr = server_socket.accept()
    print(f"[SERVER] Connected to Spark at {addr}")

    try:
        # Sample data to send to the stream
        messages = ["hello spark", "pyspark is powerful", "hello world", "streaming data"]
        for msg in messages:
            time.sleep(4) # Wait 4 seconds between each send
            conn.sendall((msg + "\n").encode())
            print(f"[SERVER] Sent: {msg}")
    except Exception as e:
        print(f"[SERVER] Error: {e}")
    finally:
        conn.close()
        server_socket.close()
        print("[SERVER] Data finished and connection closed.")

# Start the server in a background thread so it doesn't block the notebook
threading.Thread(target=start_mock_server, daemon=True).start()

[SERVER] Waiting for Spark to connect...


In [3]:
# 1. Read from the socket
lines = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# 2. Transform: Split lines into words and count them
words = lines.select(explode(split(lines.value, " ")).alias("word"))
word_counts = words.groupBy("word").count()

# 3. Output results to the console (the notebook output area)
# Using 'complete' mode to show the updated table
query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

print("[SPARK] Stream started! Watch the output below for the Batch tables...")

# 4. Wait for the stream to process the data
try:
    query.awaitTermination(timeout=30) # Runs for 30 seconds
except KeyboardInterrupt:
    print("[SPARK] Stopping stream...")
finally:
    query.stop()

[SPARK] Stream started! Watch the output below for the Batch tables...
[SERVER] Connected to Spark at ('127.0.0.1', 56692)
[SERVER] Sent: hello spark
[SERVER] Sent: pyspark is powerful
[SERVER] Sent: hello world
[SERVER] Sent: streaming data
[SERVER] Data finished and connection closed.
